## Functions & Libraries

In [0]:
#Call libraries
from pyspark.sql import functions as F
import pandas as pd

In [0]:
#Funtion to return valid candidates for an empty Sudoku cell
def get_cell_candidates(grid, row, col):
    """ Return valid candidates for an empty Sudoku cell. """

    if grid[row][col] != 0:
        return []

    row_values = set(grid[row])
    column_values = {grid[r][col] for r in range(9)}

    box_row = (row // 3) * 3
    box_col = (col // 3) * 3

    box_values = {
        grid[r][c]
        for r in range(box_row, box_row + 3)
        for c in range(box_col, box_col + 3)
    }

    used_numbers = row_values | column_values | box_values

    return sorted(set(range(1, 10)) - used_numbers)

#Funtion to apply the Naked Single technique
def solve_naked_singles(grid):
    """ Apply Naked Single moves until no further progress is possible. """

    board = [row.copy() for row in grid]
    naked_single_steps = 0

    while True:
        progress = False

        for row in range(9):
            for col in range(9):
                if board[row][col] == 0:
                    candidates = get_cell_candidates(board, row, col)

                    if len(candidates) == 1: #If there is exactly one candidate, it fills the cell and counts the operation
                        board[row][col] = candidates[0]
                        naked_single_steps += 1
                        progress = True

        if not progress:
            break

    solved = all(
        board[row][col] != 0
        for row in range(9)
        for col in range(9)
    )

    return {
        "board": board,
        "naked_single_steps": naked_single_steps,
        "solved": solved,
        "remaining_empty_cells": sum(
            cell == 0
            for row in board
            for cell in row
        )
    }

    #Notes:
        #If solved = True, it means the Sudoku has been completely solved using this technique
        #If solved = False, it means we will need other strategies to continue the solution process

## Problem-solving technique

In [0]:
#Solvers:
    #naked_single_steps -> Entries made using the "Naked Single" technique -> A cell where only one candidate is possible
    #hidden_single_steps -> Entries made using the "Hidden Single" technique -> A number that can only occupy one position within a given unit
    #backtracking_nodes -> The number of nodes explored during the search -> exploratory attempts and subsequent backtracks, used when logical techniques are insufficient to solve the puzzle
    #solver_iterations -> Cycles required to apply the strategies

### Naked Single 

In [0]:
#Naked Single technique test one case
gold_df = spark.table("workspace.sudoku_gold.game_features")
sample_puzzle = gold_df.select("quizzes").first()["quizzes"]

grid = [
    [int(sample_puzzle[row * 9 + col]) for col in range(9)]
    for row in range(9)
]

result = solve_naked_singles(grid)

print(f"Naked Single steps: {result['naked_single_steps']}") 
print(f"Remaining empty cells: {result['remaining_empty_cells']}")
print(f"Solved: {result['solved']}")


In [0]:
#Retrieve the reference solution for the same puzzle
sample_record = (
    spark.table("workspace.sudoku_bronze.raw_games")
    .filter(F.col("quizzes") == sample_puzzle)
    .select("solutions")
    .first()
)

reference_solution = sample_record["solutions"]

#Convert the reference solution into a 9x9 grid
reference_grid = [
    [int(reference_solution[row * 9 + col]) for col in range(9)]
    for row in range(9)
]

#Compare the solver output with the reference solution
solution_matches = result["board"] == reference_grid

print(f"Solver completed: {result['solved']}")
print(f"Solution matches reference: {solution_matches}")
print(f"Naked Single steps: {result['naked_single_steps']}")

In [0]:
#Naked Single technique test one hundred case
sample_df = (
    spark.table("workspace.sudoku_bronze.raw_games")
    .select("quizzes", "solutions")
    .limit(100)
    .collect()
)

test_results = []

for record in sample_df:
    puzzle = record["quizzes"]
    reference_solution = record["solutions"]

    grid = [
        [int(puzzle[row * 9 + col]) for col in range(9)]
        for row in range(9)
    ]

    reference_grid = [
        [
            int(reference_solution[row * 9 + col])
            for col in range(9)
        ]
        for row in range(9)
    ]

    result = solve_naked_singles(grid)

    test_results.append({
        "solved": result["solved"],
        "solution_matches": result["board"] == reference_grid,
        "naked_single_steps": result["naked_single_steps"],
        "remaining_empty_cells": result["remaining_empty_cells"],
    })

total_puzzles = len(test_results)
solved_count = sum(result["solved"] for result in test_results)
matching_count = sum(result["solution_matches"] for result in test_results)

print(f"Puzzles tested: {total_puzzles}")
print(f"Solved by Naked Singles: {solved_count}")
print(f"Solutions matching reference: {matching_count}")
print(f"Solve rate: {solved_count / total_puzzles:.2%}")

In [0]:
#Naked Single technique test one thousand random case
sample_df = (
    spark.table("workspace.sudoku_bronze.raw_games")
    .select("quizzes", "solutions")
    .orderBy(F.rand(seed=42))
    .limit(1000)
    .collect()
)

test_results = []

for record in sample_df:
    puzzle = record["quizzes"]
    reference_solution = record["solutions"]

    grid = [
        [int(puzzle[row * 9 + col]) for col in range(9)]
        for row in range(9)
    ]

    reference_grid = [
        [
            int(reference_solution[row * 9 + col])
            for col in range(9)
        ]
        for row in range(9)
    ]

    result = solve_naked_singles(grid)

    test_results.append({
        "solved": result["solved"],
        "solution_matches": result["board"] == reference_grid,
        "naked_single_steps": result["naked_single_steps"],
        "remaining_empty_cells": result["remaining_empty_cells"],
    })

total_puzzles = len(test_results)
solved_count = sum(item["solved"] for item in test_results)
matching_count = sum(item["solution_matches"] for item in test_results)

print(f"Puzzles tested: {total_puzzles}")
print(f"Solved by Naked Singles: {solved_count}")
print(f"Solutions matching reference: {matching_count}")
print(f"Solve rate: {solved_count / total_puzzles:.2%}")

In [0]:
#Analyze the number of candidates and the solved cells
results_df = pd.DataFrame(test_results)

print("Naked Single steps — descriptive statistics")
print(results_df["naked_single_steps"].describe())

print("\nNaked Single steps — frequency distribution")
print(
    results_df["naked_single_steps"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nRemaining empty cells — frequency distribution")
print(
    results_df["remaining_empty_cells"]
    .value_counts()
    .sort_index()
    .to_string()
)

In [0]:
#Notes:
    #The solver is working, but we need to be careful with the interpretation
    #The puzzles contain between 45 and 50 empty cells that are solved by the algorithm
    #Since all of them were solved, the `naked_single_steps` count matches the number of empty cells in these cases
    #Therefore, this variable essentially replicates information we already have in the Gold layer: `empty_cells = 81 - given_cells`

In [0]:
#Naked Single technique test ten thousand case
sample_df = (
    spark.table("workspace.sudoku_bronze.raw_games")
    .select("quizzes", "solutions")
    .sample(withReplacement=False, fraction=0.02, seed=42)
    .limit(10_000)
    .collect()
)

test_results = []

for record in sample_df:
    puzzle = record["quizzes"]
    reference_solution = record["solutions"]

    grid = [
        [int(puzzle[row * 9 + col]) for col in range(9)]
        for row in range(9)
    ]

    reference_grid = [
        [
            int(reference_solution[row * 9 + col])
            for col in range(9)
        ]
        for row in range(9)
    ]

    result = solve_naked_singles(grid)

    test_results.append({
        "solved": result["solved"],
        "solution_matches": result["board"] == reference_grid,
        "naked_single_steps": result["naked_single_steps"],
        "remaining_empty_cells": result["remaining_empty_cells"],
    })

total_puzzles = len(test_results)
solved_count = sum(item["solved"] for item in test_results)
matching_count = sum(item["solution_matches"] for item in test_results)

print(f"Puzzles tested: {total_puzzles}")
print(f"Solved by Naked Singles: {solved_count}")
print(f"Solutions matching reference: {matching_count}")
print(f"Solve rate: {solved_count / total_puzzles:.2%}")